In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
df=pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
df.head()


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['sentiment'].value_counts(normalize=True)

In [ ]:
df=df.drop_duplicates().reset_index(drop=True)
print("Shape after dedup:", df.shape)

In [ ]:
df['clean_review']=df['review']

In [ ]:
df['clean_review'].head()

In [ ]:
# to remove <br /><br />

import re
def remove_html(text):
    return re.sub(r"<.*?>","",text)

df['clean_review']=df['clean_review'].apply(remove_html)

In [ ]:
df['clean_review'].head()

In [ ]:
df['clean_review']=df['clean_review'].str.lower()

In [ ]:
df[['review','clean_review']].head()

In [ ]:
df["word_count"] = df["clean_review"].str.split().str.len()
# print(df["word_count"].describe())
df.head()

In [ ]:
# The fastest and most efficient way to remove punctuation from a string in Python is by using the str.translate() method combined with str.maketrans()
# Or use import re

def remove_punc(text):
    return re.sub(r"[^\w\s]","",text)

df['clean_review']=df['clean_review'].apply(remove_punc)

In [ ]:
df.head()

In [ ]:
# tokenisation

# "this movie was amazing"  -> ["this", "movie", "was", "amazing"]

def split(text):
    return text.split()

df['clean_review']=df['clean_review'].apply(split)

In [ ]:
df.head()

In [ ]:
df['sentiment']=df['sentiment'].map({'positive':1,'negative':0})

In [ ]:
df.head()

In [ ]:
# Why split abhi?

# Because next hum vocabulary banayenge:

from sklearn.model_selection import train_test_split

X=df['clean_review']
y=df['sentiment']

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)

In [ ]:
print(len(X_train))
print(len(X_test))

In [ ]:
print(y_train.value_counts())
print(y_test.value_counts())

In [ ]:
# Vocabulary / Word Encoding
X_train_text = X_train.apply(lambda x: " ".join(x))
X_test_text = X_test.apply(lambda x: " ".join(x))

In [ ]:
print(X_train_text)
# humne yeh isliye kra hai kyuki keras's tokenizer hi hai jo words ko weight dega pr kyuki alredy X_tarin tokens m hai toh isliye wapas text m join kra hai 

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
token=Tokenizer()

# fit only on training data
token.fit_on_texts(X_train_text)

# Training data mein kitne unique words hain.
word_index=token.word_index
print(len(word_index))


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

vocab_size = 20000

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train_text)

In [ ]:
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

In [ ]:
print(len(X_train_seq[0]))


In [ ]:
print(len(X_test_seq[0]))

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 300

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

In [ ]:
print(X_train_pad.shape)
print(X_test_pad.shape)

In [ ]:
print(X_train_pad[0])

In [ ]:
# Embedding layer in word IDs ko meaningful numerical vectors mein convert karegi.

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding

vocab_size = 20000
embedding_dim = 128

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_shape=(300,)
    )
])

sample_output = model(X_train_pad[:1])
print(sample_output)

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)